# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [2]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME", "student_your_username")
MLFLOW_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD", "your_mlflow_password")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", f"qbc12_hw02_{MLFLOW_USERNAME}")

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)


MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_your_username
Experiment ID: 6


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [3]:
DATASET_VERSION = "v1_audit_cleaned"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
elif csv_path.exists():
    feature_df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(
        f"Feature dataset not found. Expected {parquet_path} or {csv_path}. Run notebook 01 first."
    )

# TODO: load metadata if metadata_path exists.
if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    metadata = {}

print(feature_df.shape)
feature_df.head()


(10480, 33)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,...,14,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_audit_cleaned
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,...,16,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [4]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
assert TARGET_COL in feature_df.columns, f"Missing target column: {TARGET_COL}"

# TODO: create y.
y = feature_df[TARGET_COL].astype(int)

# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
clean_feature_cols = [
    col for col in feature_df.columns
    if col not in FORBIDDEN_MODEL_COLUMNS
]

# TODO: create X_clean.
X_clean = feature_df[clean_feature_cols].copy()

assert not set(FORBIDDEN_MODEL_COLUMNS).intersection(X_clean.columns)
assert len(clean_feature_cols) > 0

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)


Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64
Clean feature count: 26
['room_type', 'property_type', 'neighbourhood_name', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'host_is_superhost', 'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [5]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.
assert LEAKAGE_COLUMN in feature_df.columns, f"Missing leakage demo column: {LEAKAGE_COLUMN}"

leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]
leaky_feature_cols = [col for col in leaky_feature_cols if col != TARGET_COL]
X_leaky = feature_df[leaky_feature_cols].copy()

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)


Leaky feature count: 27
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [6]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())


Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [7]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(input_df):
    numeric_columns = input_df.select_dtypes(include=["number", "bool", "boolean"]).columns.tolist()
    categorical_columns = [col for col in input_df.columns if col not in numeric_columns]

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_columns),
            ("categorical", categorical_transformer, categorical_columns),
        ],
        remainder="drop",
    ), numeric_columns, categorical_columns


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.
preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_clean)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


Numeric columns: 23
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [8]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    # 2. convert scores to predictions using threshold
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    # 4. return metrics dict, y_pred, y_score
    y_score = get_positive_scores(model, X_test)
    y_pred = (y_score >= threshold).astype(int)

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, y_score),
        "prediction_threshold": threshold,
    }

    return metrics, y_pred, y_score


## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [9]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    # 2. save confusion_matrix.png
    # 3. save classification_report.json
    # 4. save feature_columns.json
    # 5. save dataset_metadata_snapshot.json
    safe_run_name = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in run_name)
    run_artifact_dir = ARTIFACT_DIR / safe_run_name
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(run_name)
    fig.tight_layout()
    fig.savefig(run_artifact_dir / "confusion_matrix.png", dpi=150)
    plt.close(fig)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    with open(run_artifact_dir / "classification_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    with open(run_artifact_dir / "feature_columns.json", "w", encoding="utf-8") as f:
        json.dump({"feature_columns": list(feature_cols)}, f, indent=2)

    with open(run_artifact_dir / "dataset_metadata_snapshot.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, default=str)

    return run_artifact_dir


## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [10]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)
    pipeline.fit(X_train, y_train)
    metrics, y_pred, y_score = evaluate_binary_classifier(pipeline, X_test, y_test, threshold=threshold)
    run_artifact_dir = save_run_artifacts(run_name, y_test, y_pred, feature_cols, metadata)

    params_to_log = {
        "dataset_version": DATASET_VERSION,
        "feature_count": len(feature_cols),
        "threshold": threshold,
        **model_params,
    }

    tags_to_log = {
        "homework": "hw02",
        "target": TARGET_COL,
        **tags,
    }

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params_to_log)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(tags_to_log)
        mlflow.log_artifacts(str(run_artifact_dir), artifact_path="evaluation_artifacts")
        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        run_id = run.info.run_id

    result = {
        "run_name": run_name,
        "run_id": run_id,
        **metrics,
    }
    print(result)
    return result


## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [11]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

leaky_preprocessor, leaky_numeric_cols, leaky_categorical_cols = make_preprocessor(X_leaky)
leaky_logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", leaky_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

v0_result = run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression",
    pipeline=leaky_logreg_pipeline,
    X_train=X_train_leaky,
    X_test=X_test_leaky,
    y_train=y_train_leaky,
    y_test=y_test_leaky,
    feature_cols=leaky_feature_cols,
    model_params={"model_type": "LogisticRegression", "max_iter": 1000},
    tags={"uses_leakage": "true", "clean_candidate": "false"},
)


2026/07/05 19:01:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/6/runs/dc6d1efce8264b0891750ffd751be096
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v0_leaky_logistic_regression', 'run_id': 'dc6d1efce8264b0891750ffd751be096', 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1': 1.0, 'test_roc_auc': 1.0, 'prediction_threshold': 0.5}


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [12]:
# TODO: build and log dummy baseline.
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        ("classifier", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
    ]
)

v1_result = run_mlflow_experiment(
    run_name="v1_dummy_baseline",
    pipeline=dummy_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model_type": "DummyClassifier", "strategy": "most_frequent"},
    tags={"uses_leakage": "false", "clean_candidate": "true"},
)


2026/07/05 19:01:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/6/runs/520161de1e7f416698ba69988f8875fe
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v1_dummy_baseline', 'run_id': '520161de1e7f416698ba69988f8875fe', 'test_accuracy': 0.762881679389313, 'test_precision': 0.762881679389313, 'test_recall': 1.0, 'test_f1': 0.8654939106901218, 'test_roc_auc': 0.5, 'prediction_threshold': 0.5}


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [13]:
# TODO: build and log clean LogisticRegression.
clean_logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

v2_result = run_mlflow_experiment(
    run_name="v2_clean_logistic_regression",
    pipeline=clean_logreg_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model_type": "LogisticRegression", "max_iter": 1000},
    tags={"uses_leakage": "false", "clean_candidate": "true"},
)


2026/07/05 19:01:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/6/runs/ffdcb7b36cba400da9b2b09e090d29c4
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v2_clean_logistic_regression', 'run_id': 'ffdcb7b36cba400da9b2b09e090d29c4', 'test_accuracy': 0.976145038167939, 'test_precision': 0.9760295021511985, 'test_recall': 0.9931207004377736, 'test_f1': 0.9845009299442034, 'test_roc_auc': 0.9892249054049123, 'prediction_threshold': 0.5}


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [14]:
# TODO: build and log class-weighted LogisticRegression.
balanced_logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]
)

v3_result = run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression",
    pipeline=balanced_logreg_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model_type": "LogisticRegression", "max_iter": 1000, "class_weight": "balanced"},
    tags={"uses_leakage": "false", "clean_candidate": "true"},
)


2026/07/05 19:02:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/6/runs/f9a8a096a00943498e8478f44c0bb602
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v3_balanced_logistic_regression', 'run_id': 'f9a8a096a00943498e8478f44c0bb602', 'test_accuracy': 0.9770992366412213, 'test_precision': 0.9813780260707635, 'test_recall': 0.9887429643527205, 'test_f1': 0.9850467289719627, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.5}


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [15]:
# TODO: log threshold-tuning runs.
threshold_results = []
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    threshold_pipeline = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(X_clean)[0]),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    )
    threshold_results.append(
        run_mlflow_experiment(
            run_name=f"v4_balanced_logistic_threshold_{threshold:.2f}",
            pipeline=threshold_pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            feature_cols=clean_feature_cols,
            model_params={
                "model_type": "LogisticRegression",
                "max_iter": 1000,
                "class_weight": "balanced",
                "threshold_tuning": True,
            },
            tags={"uses_leakage": "false", "clean_candidate": "true", "threshold_tuning": "true"},
            threshold=threshold,
        )
    )

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df.sort_values("test_f1", ascending=False)


2026/07/05 19:02:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold_0.30 at: http://185.50.38.163:33014/#/experiments/6/runs/b1510031c71c40caad8e4b2bf9bd0a27
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v4_balanced_logistic_threshold_0.30', 'run_id': 'b1510031c71c40caad8e4b2bf9bd0a27', 'test_accuracy': 0.9770992366412213, 'test_precision': 0.977818853974122, 'test_recall': 0.9924953095684803, 'test_f1': 0.9851024208566108, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.3}


2026/07/05 19:02:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold_0.40 at: http://185.50.38.163:33014/#/experiments/6/runs/5b8063a53f934c0ea04cad47a594dde6
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


...

{'run_name': 'v4_balanced_logistic_threshold_0.60', 'run_id': 'e6677c85356941a3b1019b51d3cb83dd', 'test_accuracy': 0.9770992366412213, 'test_precision': 0.9825762289981331, 'test_recall': 0.9874921826141339, 'test_f1': 0.9850280723643169, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.6}


2026/07/05 19:04:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold_0.70 at: http://185.50.38.163:33014/#/experiments/6/runs/1bbd407097054976a721694c79fa25e1
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v4_balanced_logistic_threshold_0.70', 'run_id': '1bbd407097054976a721694c79fa25e1', 'test_accuracy': 0.9790076335877863, 'test_precision': 0.9850280723643169, 'test_recall': 0.9874921826141339, 'test_f1': 0.9862585883822611, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.7}


,run_name,run_id,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,prediction_threshold
4,v4_balanced_logistic_threshold_0.70,1bbd407097054976a721694c79fa25e1,0.979008,0.985028,0.987492,0.986259,0.990559,0.7
1,v4_balanced_logistic_threshold_0.40,5b8063a53f934c0ea04cad47a594dde6,0.978053,0.980210,0.991245,0.985697,0.990559,0.4
0,v4_balanced_logistic_threshold_0.30,b1510031c71c40caad8e4b2bf9bd0a27,0.977099,0.977819,0.992495,0.985102,0.990559,0.3
2,v4_balanced_logistic_threshold_0.50,cd8989d193cc4b6a882f398eacffa056,0.977099,0.981378,0.988743,0.985047,0.990559,0.5
3,v4_balanced_logistic_threshold_0.60,e6677c85356941a3b1019b51d3cb83dd,0.977099,0.982576,0.987492,0.985028,0.990559,0.6


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [16]:
# TODO: build and log RandomForestClassifier.
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

v5_result = run_mlflow_experiment(
    run_name="v5_random_forest",
    pipeline=random_forest_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={
        "model_type": "RandomForestClassifier",
        "n_estimators": 200,
        "min_samples_leaf": 2,
        "class_weight": "balanced_subsample",
    },
    tags={"uses_leakage": "false", "clean_candidate": "true"},
)


2026/07/05 19:04:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/6/runs/a69f97c9625a461cbf534dd6e2dabfa9
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/6


{'run_name': 'v5_random_forest', 'run_id': 'a69f97c9625a461cbf534dd6e2dabfa9', 'test_accuracy': 0.9833015267175572, 'test_precision': 0.986318407960199, 'test_recall': 0.991869918699187, 'test_f1': 0.9890863735578422, 'test_roc_auc': 0.9937687412781882, 'prediction_threshold': 0.5}


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [17]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# TODO: retrieve MLflow runs for this experiment and create a comparison table.
if experiment is None:
    raise ValueError(f"Experiment does not exist: {EXPERIMENT_NAME}")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_f1 DESC"],
)

comparison_columns = [
    "run_id",
    "tags.mlflow.runName",
    "tags.uses_leakage",
    "tags.clean_candidate",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.test_f1",
    "metrics.test_roc_auc",
    "metrics.prediction_threshold",
    "params.model_type",
    "params.feature_count",
]
comparison_columns = [col for col in comparison_columns if col in runs_df.columns]
comparison_df = runs_df[comparison_columns].copy()
comparison_df = comparison_df.sort_values("metrics.test_f1", ascending=False)

comparison_df


,run_id,tags.mlflow.runName,tags.uses_leakage,tags.clean_candidate,metrics.test_accuracy,metrics.test_precision,metrics.test_recall,metrics.test_f1,metrics.test_roc_auc,metrics.prediction_threshold,params.model_type,params.feature_count
0,dc6d1efce8264b0891750ffd751be096,v0_leaky_logistic_regression,true,false,1.000000,1.000000,1.000000,1.000000,1.000000,0.5,LogisticRegression,27
1,3eaf557d4df94e4dbb2ad6aa11d641ef,v0_leaky_logistic_regression,true,false,1.000000,1.000000,1.000000,1.000000,1.000000,0.5,LogisticRegression,27
2,a69f97c9625a461cbf534dd6e2dabfa9,v5_random_forest,false,true,0.983302,0.986318,0.991870,0.989086,0.993769,0.5,RandomForestClassifier,26
3,20f89f8d6e5c4b848137e811a6eeec51,v5_random_forest,false,true,0.983302,0.986318,0.991870,0.989086,0.993769,0.5,RandomForestClassifier,26
4,1bbd407097054976a721694c79fa25e1,v4_balanced_logistic_threshold_0.70,false,true,0.979008,0.985028,0.987492,0.986259,0.990559,0.7,LogisticRegression,26
5,d1505af3bee949859bfb3c23abd585c0,v4_balanced_logistic_threshold_0.70,false,true,0.979008,0.985028,0.987492,0.986259,0.990559,0.7,LogisticRegression,26
6,5b8063a53f934c0ea04cad47a594dde6,v4_balanced_logistic_threshold_0.40,false,true,0.978053,0.980210,0.991245,0.985697,0.990559,0.4,LogisticRegression,26
7,938c0d360dd04e04960f889091725b7b,v4_balanced_logistic_threshold_0.40,false,true,0.978053,0.980210,0.991245,0.985697,0.990559,0.4,LogisticRegression,26
8,b1510031c71c40caad8e4b2bf9bd0a27,v4_balanced_logistic_threshold_0.30,false,true,0.977099,0.977819,0.992495,0.985102,0.990559,0.3,LogisticRegression,26
9,da46ae81a3c2478e8fb10d2af77e38d8,v4_balanced_logistic_threshold_0.30,false,true,0.977099,0.977819,0.992495,0.985102,0.990559,0.3,LogisticRegression,26


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [18]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
clean_runs = comparison_df[
    (comparison_df.get("tags.uses_leakage", "false") != "true")
    & (comparison_df.get("tags.clean_candidate", "true") == "true")
].copy()

if clean_runs.empty:
    raise ValueError("No clean candidate runs found. Run the clean experiments first.")

BEST_RUN_ID = clean_runs.sort_values("metrics.test_f1", ascending=False).iloc[0]["run_id"]

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)


Selected best run: a69f97c9625a461cbf534dd6e2dabfa9


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [19]:
# TODO: replace this text.
final_explanation = f"""
I built a leakage-aware experiment set for the Airbnb high-demand proxy task.
The v0 run intentionally includes future_available_rate_30d to demonstrate leakage and should not be used for serving.
The clean candidate runs exclude listing identifiers, audit fields, future-window columns, and the target.

The selected serving run is {BEST_RUN_ID}. It was chosen from clean_candidate runs by highest test F1 score in the MLflow comparison table.
For production-style serving, HW03 should load runs:/{BEST_RUN_ID}/model and validate incoming payloads against the exact clean feature list.
"""

print(final_explanation)



I built a leakage-aware experiment set for the Airbnb high-demand proxy task.
The v0 run intentionally includes future_available_rate_30d to demonstrate leakage and should not be used for serving.
The clean candidate runs exclude listing identifiers, audit fields, future-window columns, and the target.

The selected serving run is a69f97c9625a461cbf534dd6e2dabfa9. It was chosen from clean_candidate runs by highest test F1 score in the MLflow comparison table.
For production-style serving, HW03 should load runs:/a69f97c9625a461cbf534dd6e2dabfa9/model and validate incoming payloads against the exact clean feature list.

